In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import yaml


In [1]:
# Resolve paths from the project config so the dataset location is defined in one place.
PROJECT_ROOT = Path.cwd().resolve().parent
with (PROJECT_ROOT / 'config' / 'config.yaml').open(encoding='utf-8') as config_file:
    config = yaml.safe_load(config_file)

DATA_DIR = Path(config['data_ingestion']['source_dir']).expanduser()
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')

class_counts = {
    class_dir.name: sum(path.is_file() for path in class_dir.iterdir())
    for class_dir in sorted(DATA_DIR.iterdir())
    if class_dir.is_dir()
}
pd.DataFrame(class_counts.items(), columns=['class_name', 'image_count'])


'/Users/Shared/ThoraxGuard/research'

In [4]:
# Deterministic 80/20 split. Images are decoded as RGB and resized for CNN input.
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.20,
    subset='training',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.20,
    subset='validation',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='rgb',
)

class_names = train_ds.class_names
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

train_batch, train_labels = next(iter(train_ds))
print(f'Classes: {class_names}')
print(f'Training batch: images={train_batch.shape}, labels={train_labels.shape}')
print(f'Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}')